
# Relation Network (RN) – Few-Shot Learning (PyTorch, MNIST)

Este notebook implementa uma **Relation Network** com:
- **Episodic training** (C-way K-shot)
- **Embedding Module** (CNN, pesos compartilhados)
- **Relation Module** (CNN + MLP) produzindo *relation scores* ∈ [0, 1]
- **Loss MSE** (como no paper original) e opção **BCE**
- **Avaliação** em episódios com classes não vistas no treino

> Para um teste rápido, reduza `episodes` (ex.: 50). Para melhor desempenho, aumente.


In [ ]:
!pip install torch torchvision --quiet


In [ ]:
import random, math
from typing import Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


## Embedding Module (fϕ) – CNN de 4 blocos Conv-BN-ReLU-Pool

In [ ]:

def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

class EmbeddingNet(nn.Module):
    """
    4 x [Conv(3x3,64) -> BN -> ReLU -> MaxPool(2x2)]
    Para MNIST 28x28: 28 -> 14 -> 7 -> 3 -> 1 (dimensões espaciais)
    Saída: (B, 64, 1, 1) -> flatten ~ 64-dim
    """
    def __init__(self, in_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(in_channels, 64),
            conv_block(64, 64),
            conv_block(64, 64),
            conv_block(64, 64),
        )

    def forward(self, x):
        return self.net(x)  # (B, 64, H', W') tipicamente (B,64,1,1) p/ MNIST


## Relation Module (gϕ) – CNN + MLP que aprende a função de comparação

In [ ]:
class RelationModule(nn.Module):
    """
    CNN + MLP que recebe pares concatenados (Q×S, 128, 1, 1) e gera relation scores.
    Projetado para MNIST (embedding -> 64x1x1; concat -> 128x1x1).
    """
    def __init__(self, in_channels=128, mid_channels=64, fc_hidden=8):
        super().__init__()
        self.g = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
        )

        # Para MNIST após 4 pools: H'=W'=1
        flat_sz = mid_channels * 1 * 1

        self.fc1 = nn.Linear(flat_sz, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden, 1)

    def forward(self, x):
        out = self.g(x)                 # (B, mid_channels, 1, 1)
        out = out.view(out.size(0), -1) # (B, mid_channels)
        out = F.relu(self.fc1(out))
        out = torch.sigmoid(self.fc2(out))
        return out


## Dados – Episódios N-way K-shot a partir do MNIST

In [ ]:

def split_classes(num_classes: int, split_ratio: float = 0.7):
    idxs = list(range(num_classes))
    random.shuffle(idxs)
    k = int(split_ratio * num_classes)
    return idxs[:k], idxs[k:]

def group_by_class(ds):
    by_class = {}
    for img, y in ds:
        y = int(y)
        by_class.setdefault(y, []).append((img, y))
    return by_class

class EpisodicFromDict:
    def __init__(self, by_class_dict):
        self.by_class = by_class_dict
        self.classes = sorted(self.by_class.keys())

    def sample_episode(self, ways: int, shots: int, queries: int):
        cls_ids = random.sample(self.classes, ways)
        support_imgs, support_lbls, query_imgs, query_lbls = [], [], [], []
        for epi_idx, cls in enumerate(cls_ids):
            pool = self.by_class[cls]
            chosen = random.sample(pool, shots + queries)
            s = chosen[:shots]
            q = chosen[shots:]
            for img, _ in s:
                support_imgs.append(img)
                support_lbls.append(epi_idx)  # remapeia p/ [0..ways-1]
            for img, _ in q:
                query_imgs.append(img)
                query_lbls.append(epi_idx)
        support_x = torch.stack(support_imgs, dim=0)
        support_y = torch.tensor(support_lbls, dtype=torch.long)
        query_x = torch.stack(query_imgs, dim=0)
        query_y = torch.tensor(query_lbls, dtype=torch.long)
        return support_x, support_y, query_x, query_y

# Carrega MNIST e divide classes em treino/teste (ex.: 7/3)
transform = T.Compose([T.ToTensor()])
full_train = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
full_test  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_classes, test_classes = split_classes(10, split_ratio=0.7)
train_by_class = {c: [] for c in train_classes}
test_by_class  = {c: [] for c in test_classes}

for img, y in full_train:
    if int(y) in train_classes:
        train_by_class[int(y)].append((img, int(y)))

for img, y in full_test:
    if int(y) in test_classes:
        test_by_class[int(y)].append((img, int(y)))

train_data = EpisodicFromDict(train_by_class)
test_data  = EpisodicFromDict(test_by_class)
print("Train classes:", train_classes, "| Test classes:", test_classes)


## Forward por pares (query × support) e agregação por classe

In [ ]:

def rn_relation_scores(embedding_net, relation_net, support_x, query_x):
    # Envia p/ device
    support_x = support_x.to(device)
    query_x = query_x.to(device)

    # Embeddings compartilhados
    z_support = embedding_net(support_x)  # (S, 64, H', W')
    z_query = embedding_net(query_x)      # (Q, 64, H', W')

    S = z_support.size(0)
    Q = z_query.size(0)

    # Pareia tudo: concat canal -> (Q*S, 128, H', W')
    zq_exp = z_query.unsqueeze(1).expand(Q, S, *z_query.shape[1:])
    zs_exp = z_support.unsqueeze(0).expand(Q, S, *z_support.shape[1:])
    pair = torch.cat([zq_exp, zs_exp], dim=2).contiguous()
    pair = pair.view(Q*S, 128, z_query.size(2), z_query.size(3))

    scores = relation_net(pair).view(Q, S)  # (Q,S) em [0,1]
    return scores

def aggregate_scores(scores, support_labels, ways, shots):
    Q, S = scores.size()
    support_labels = support_labels.to(scores.device)
    class_scores = torch.zeros(Q, ways, device=scores.device)
    for c in range(ways):
        mask = (support_labels == c).to(scores.dtype).unsqueeze(0)  # (1,S)
        num_c = mask.sum(dim=1, keepdim=True).clamp(min=1.)
        class_scores[:, c] = (scores * mask).sum(dim=1) / num_c.squeeze(1)
    pred = class_scores.argmax(dim=1)
    return pred, class_scores


## Treinamento (MSE por par, Adam)

In [ ]:

def train_rn(embedding_net, relation_net, episodic_data, ways=5, shots=1, queries=5, episodes=200, lr=1e-3, loss_type="mse"):
    embedding_net.to(device)
    relation_net.to(device)
    params = list(embedding_net.parameters()) + list(relation_net.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    history = []
    for ep in range(1, episodes+1):
        support_x, support_y, query_x, query_y = episodic_data.sample_episode(ways, shots, queries)
        support_x, support_y = support_x.to(device), support_y.to(device)
        query_x, query_y = query_x.to(device), query_y.to(device)

        opt.zero_grad()

        scores = rn_relation_scores(embedding_net, relation_net, support_x, query_x)  # (Q,S)

        # Rótulos por par (1 se mesma classe, senão 0)
        Q, S = scores.size()
        pair_labels = torch.zeros(Q, S, device=device)
        for q in range(Q):
            for s in range(S):
                pair_labels[q, s] = 1.0 if support_y[s].item() == query_y[q].item() else 0.0

        if loss_type == "mse":
            loss = F.mse_loss(scores, pair_labels)
        else:
            loss = F.binary_cross_entropy(scores, pair_labels)

        loss.backward()
        opt.step()

        pred, _ = aggregate_scores(scores.detach(), support_y, ways, shots)
        acc = (pred == query_y).float().mean().item()

        history.append((ep, float(loss.item()), float(acc)))
        if ep % max(1, episodes//10) == 0:
            print(f"[Ep {ep:4d}] loss={loss.item():.4f}  acc={acc*100:.1f}%")

    return history


## Avaliação em episódios de teste (classes não vistas no treino)

In [ ]:

@torch.no_grad()
def eval_rn(embedding_net, relation_net, episodic_data, ways=5, shots=1, queries=5, episodes=50):
    embedding_net.to(device)
    relation_net.to(device)
    accs = []
    for _ in range(episodes):
        support_x, support_y, query_x, query_y = episodic_data.sample_episode(ways, shots, queries)
        support_x, support_y = support_x.to(device), support_y.to(device)
        query_x, query_y = query_x.to(device), query_y.to(device)
        scores = rn_relation_scores(embedding_net, relation_net, support_x, query_x)
        pred, _ = aggregate_scores(scores, support_y, ways, shots)
        accs.append((pred == query_y).float().mean().item())
    return sum(accs)/len(accs)


## Execução: treinar e avaliar (ajuste `episodes` para teste rápido)

In [ ]:

ways, shots, queries = 3, 1, 5
episodes_train, episodes_test = 500, 20   # aumente para melhor desempenho

f_phi = EmbeddingNet(in_channels=1)
g_phi = RelationModule(in_channels=128, mid_channels=64, fc_hidden=8)


hist = train_rn(f_phi, g_phi, train_data, ways, shots, queries, episodes=episodes_train, lr=1e-3, loss_type="mse")
test_acc = eval_rn(f_phi, g_phi, test_data, ways, shots, queries, episodes=episodes_test)

print(f"Test few-shot accuracy over {episodes_test} episodes: {test_acc*100:.2f}%")
